# CodeGuard AI — Fine-Tuning CodeBERT on CodeXGLUE Defect Detection

This notebook demonstrates fine-tuning `microsoft/codebert-base` on the CodeXGLUE defect detection dataset (Devign: C code from FFmpeg and QEMU) using a free Google Colab GPU (T4).

The trained model export can be copied to `codeguard/artifacts/codebert/` to provide an optional extra signal to `predict.py`.

In [ ]:
!pip install -q transformers datasets accelerate torch evaluate scikit-learn

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import numpy as np
import evaluate
import json
from pathlib import Path

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
# 1. Load CodeXGLUE defect detection dataset
dataset = load_dataset("google/code_x_glue_cc_defect_detection")
print(dataset)

# 2. Tokenize with CodeBERT
model_name = "microsoft/codebert-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_fn(examples):
    return tokenizer(examples["func"], padding="max_length", truncation=True, max_length=256)

tokenized = dataset.map(tokenize_fn, batched=True)
tokenized = tokenized.rename_column("target", "labels")

In [ ]:
# 3. Set up model & evaluation metric
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=preds, references=labels)["accuracy"]
    f1 = f1_metric.compute(predictions=preds, references=labels)["f1"]
    return {"accuracy": acc, "f1": f1}

In [ ]:
# 4. Training arguments
training_args = TrainingArguments(
    output_dir="./codebert_defect",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    fp16=torch.cuda.is_available(),
    logging_steps=100,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    compute_metrics=compute_metrics,
)

# 5. Train
trainer.train()

# 6. Evaluate on test set
test_results = trainer.evaluate(tokenized["test"])
print("Test Results:", test_results)

In [ ]:
# 7. Save export
export_dir = Path("./codebert_export")
export_dir.mkdir(parents=True, exist_ok=True)
model.save_pretrained(export_dir)
tokenizer.save_pretrained(export_dir)
print(f"Saved CodeBERT export to {export_dir}")